# Practice: experiments you can trust
*AI from scratch, toolkit 6* · lesson: https://ihor.sh/ai/experiments.html

Make a PyTorch run repeat exactly, measure the seed lottery, log runs and compare two settings over several seeds
with a test, fingerprint a dataset, and save a model together with the story of how it was made.
Replace each `...` under `# YOUR CODE HERE`, run the cell, then the check under it; `hint(n)` and `answer(n)` are
there when you're stuck.

In [ ]:
# Helpers: run this cell first. check() marks an answer ✓ or ✗; stuck? hint(n) gives the next hint for
# exercise n, answer(n) shows a solution.
import base64 as _b64
_HINTS = {1: ['Two more calls, each given seed.', 'np.random.seed(seed) and torch.manual_seed(seed), one per line'], 2: ['A list comprehension over range(8).', 'accs = [train(seed=s)[1] for s in range(8)]'], 3: ['Two loops: for h in [16, 32]: and inside it for s in range(5):', 'Inside: acc = train(hidden=h, seed=s)[1], then log_run({"hidden": h, "seed": s}, test_acc=acc)'], 4: ['a = runs[runs["hidden"] == 16]["test_acc"], and b the same for 32.', 'p = stats.ttest_ind(a, b, equal_var=False).pvalue'], 5: ['Hash data bytes with hashlib.sha256, convert to hexadecimal string, and slice the first 16 characters.', 'fp = hashlib.sha256(data).hexdigest()[:16]'], 6: ['checkpoint = {"state_dict": best.state_dict(), "config": config, "data": fp, "metrics": {"test_acc": best_acc}}', 'Then torch.save(checkpoint, "best.pt")']}
_ANSWERS = {1: 'bnAucmFuZG9tLnNlZWQoc2VlZCkKdG9yY2gubWFudWFsX3NlZWQoc2VlZCk=', 2: 'YWNjcyA9IFt0cmFpbihzZWVkPXMpWzFdIGZvciBzIGluIHJhbmdlKDgpXQ==', 3: 'Zm9yIGggaW4gWzE2LCAzMl06CiAgICBmb3IgcyBpbiByYW5nZSg1KToKICAgICAgICBhY2MgPSB0cmFpbihoaWRkZW49aCwgc2VlZD1zKVsxXQogICAgICAgIGxvZ19ydW4oeyJoaWRkZW4iOiBoLCAic2VlZCI6IHN9LCB0ZXN0X2FjYz1hY2Mp', 4: 'YSA9IHJ1bnNbcnVuc1siaGlkZGVuIl0gPT0gMTZdWyJ0ZXN0X2FjYyJdCmIgPSBydW5zW3J1bnNbImhpZGRlbiJdID09IDMyXVsidGVzdF9hY2MiXQpwID0gc3RhdHMudHRlc3RfaW5kKGEsIGIsIGVxdWFsX3Zhcj1GYWxzZSkucHZhbHVl', 5: 'ZnAgPSBoYXNobGliLnNoYTI1NihkYXRhKS5oZXhkaWdlc3QoKVs6MTZd', 6: 'Y2hlY2twb2ludCA9IHsic3RhdGVfZGljdCI6IGJlc3Quc3RhdGVfZGljdCgpLCAiY29uZmlnIjogY29uZmlnLCAiZGF0YSI6IGZwLCAibWV0cmljcyI6IHsidGVzdF9hY2MiOiBiZXN0X2FjY319CnRvcmNoLnNhdmUoY2hlY2twb2ludCwgImJlc3QucHQiKQ=='}
_given = {}

def check(n, test, msg=""):
    try:
        ok = bool(test())
    except NameError as e:
        return print("✗ %s: run the exercise cell above first." % e)
    except Exception as e:
        if "NoneType" in str(e) or "ellipsis" in str(e):
            return print("✗ Not answered yet: replace the ... in exercise %d and run that cell. Stuck? hint(%d)" % (n, n))
        ok, msg = False, msg + " (the check stopped with %s: %s)" % (type(e).__name__, e)
    print("✓ Right." if ok else "✗ Not yet. " + msg + "\n  Stuck? hint(%d) for a hint, answer(%d) for a solution." % (n, n))

def hint(n):
    hints, i = _HINTS.get(n, []), _given.get(n, 0)
    if not hints:
        return print("No hints for this one: answer(%d) shows a solution." % n)
    print("Hint %d of %d: %s" % (min(i, len(hints) - 1) + 1, len(hints), hints[min(i, len(hints) - 1)]))
    _given[n] = i + 1

def answer(n):
    print(_b64.b64decode(_ANSWERS[n]).decode())
    print("\n(Put this in place of the ..., run it, then close it and try writing it yourself.)")


In [ ]:
import hashlib, json, random, time, urllib.request
import numpy as np
import pandas as pd
import torch
from torch import nn
import torch.nn.functional as F
from scipy import stats
from sklearn.datasets import load_digits

X, y = load_digits(return_X_y=True)
X = torch.tensor(X / 16, dtype=torch.float32); y = torch.tensor(y)
split = torch.randperm(len(X), generator=torch.Generator().manual_seed(0))
tr, te = split[:1260], split[1260:]

## 1. Seed everything
`set_seed(seed)` seeds Python's `random` already. Add the other two: NumPy (`np.random.seed`) and PyTorch
(`torch.manual_seed`).

In [ ]:
def set_seed(seed):
    random.seed(seed)
    # YOUR CODE HERE (stuck? hint(1), then answer(1))
    ...

In [ ]:
def train(hidden=16, lr=0.01, epochs=60, seed=0):
    "A small network on the digits, trained in minibatches of 64 shuffled by the seed. Returns (model, test accuracy)."
    set_seed(seed)
    model = nn.Sequential(nn.Linear(64, hidden), nn.ReLU(), nn.Linear(hidden, 10))
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for epoch in range(epochs):
        order = torch.randperm(len(tr))
        for i in range(0, len(tr), 64):
            idx = tr[order[i:i + 64]]
            opt.zero_grad(); F.cross_entropy(model(X[idx]), y[idx]).backward(); opt.step()
    with torch.no_grad():
        return model, (model(X[te]).argmax(1) == y[te]).float().mean().item()

m1, a1 = train(seed=3); m2, a2 = train(seed=3)
print("seed 3 twice:", a1, a2)
check(1, lambda: torch.equal(m1[0].weight, m2[0].weight) and a1 == a2, "the same seed gives exactly the same weights and accuracy")

## 2. The seed lottery
Train the default model with seeds 0 to 7 and set `accs` to the list of their test accuracies (`train(seed=s)[1]`).
**Before running, estimate** the gap between the best and the worst seed, in accuracy points.

In [ ]:
accs = None
# YOUR CODE HERE (stuck? hint(2), then answer(2))
...
print(np.round(accs, 3) if accs is not None else None)

In [ ]:
check(2, lambda: len(accs) == 8 and max(accs) - min(accs) > 0.003, "eight different accuracies from one setting: that spread is the noise any comparison must beat")
if accs is not None:
    print("spread:", round(100 * (max(accs) - min(accs)), 1), "points between the best and worst seed")

## 3. Log every run
`log_run(config, **metrics)` appends one JSON line to `LOG` (given). Log 5 seeds of two settings, `hidden=16` and
`hidden=32`: for each, call `train(hidden=h, seed=s)` and `log_run({"hidden": h, "seed": s}, test_acc=acc)`.
Then `runs` (given) turns the log into a table.

In [ ]:
LOG = []
def log_run(config, **metrics):
    LOG.append(json.dumps({"time": time.strftime("%H:%M:%S"), **config, **metrics}))

# YOUR CODE HERE (stuck? hint(3), then answer(3))
...
runs = pd.DataFrame([json.loads(line) for line in LOG])
runs.groupby("hidden")["test_acc"].agg(["mean", "std", "count"]) if len(runs) else runs

In [ ]:
check(3, lambda: len(runs) == 10 and set(runs["hidden"]) == {16, 32} and runs.groupby("hidden").size().tolist() == [5, 5],
      "10 rows: 5 seeds for each of the two settings")

## 4. Is 32 better than 16, beyond luck?
Compare the two groups of 5 accuracies with Welch's t-test from Probability 4:
`stats.ttest_ind(a, b, equal_var=False)`. Set `p` to its p-value.

In [ ]:
p = None
# YOUR CODE HERE (stuck? hint(4), then answer(4))
...
print("p =", p)

In [ ]:
check(4, lambda: 0 <= p <= 1 and np.isclose(p, stats.ttest_ind(runs[runs["hidden"] == 16]["test_acc"], runs[runs["hidden"] == 32]["test_acc"], equal_var=False).pvalue),
      "Welch's test on the two groups of five; with one run each there would be nothing to test at all")

## 5. Fingerprint the data
Set `fp` to the first 16 characters of the SHA-256 hash of the outages file's bytes:
`hashlib.sha256(data).hexdigest()[:16]`.

In [ ]:
data = urllib.request.urlopen("https://raw.githubusercontent.com/ikruchynskyi/ihor.sh/main/ai/data/outages.csv").read()
fp = None
# YOUR CODE HERE (stuck? hint(5), then answer(5))
...
print(fp)

In [ ]:
check(5, lambda: fp == hashlib.sha256(data).hexdigest()[:16] and fp != hashlib.sha256(data.replace(b"Repair", b"repair", 1)).hexdigest()[:16],
      "16 hex characters; change one letter of the file and they all change")

## 6. Save a model with its story
Save `best` (trained below) as one file: a dict with its `"state_dict"`, its `"config"`, the data fingerprint
`"data"`, and its `"metrics"`, using `torch.save(checkpoint, "best.pt")`. Then the given cell loads it into a fresh
model and checks it predicts exactly the same.

In [ ]:
config = {"hidden": 32, "lr": 0.01, "epochs": 60, "seed": 0}
best, best_acc = train(**config)
# YOUR CODE HERE (stuck? hint(6), then answer(6))
...

In [ ]:
def reload(path="best.pt"):
    ck = torch.load(path)
    model = nn.Sequential(nn.Linear(64, ck["config"]["hidden"]), nn.ReLU(), nn.Linear(ck["config"]["hidden"], 10))
    model.load_state_dict(ck["state_dict"])
    return model, ck
check(6, lambda: torch.equal(reload()[0](X[te]).argmax(1), best(X[te]).argmax(1)) and reload()[1]["data"] == fp and reload()[1]["config"] == config,
      "the reloaded model predicts exactly as the saved one, and the file says how it was made")

## Next
Back to the path: https://ihor.sh/ai/ · Made With ML on experiment tracking: https://madewithml.com/